# Decorators

---

### Table of Contents
1. What are Decorators?
2. The Anatomy of a Decorator
3. Applying Decorators
4. Preserving Metadata with `functools.wraps`
5. Decorators with Arguments

---

## 1. What are Decorators?
- A decorator is a function that takes another function as input, adds some functionality (by "wrapping" it), and returns the modified function. 
- It allows you to extend the behavior of functions without permanently modifying their code.
- Decorators are a practical application of Higher-Order Functions and Closures.
- The syntax for using them is `@decorator_name`.

---

## 2. The Anatomy of a Decorator
- A decorator is a callable that returns a callable.
- Let's build one from scratch to understand the pattern.

In [1]:
def logger_decorator(original_function):
    """A decorator that logs when a function is called."""

    # A decorator returns a new function, usually called a "wrapper".
    # This wrapper function is a closure; it has access to `original_function`.
    def wrapper(*args, **kwargs):
        # `*args` and `**kwargs` allow our wrapper to accept any arguments,
        # making it compatible with any function.

        print(f"--- Calling function: '{original_function.__name__}' ---")

        # Call the original function, passing along the arguments.
        result = original_function(*args, **kwargs)

        print(f"--- Function '{original_function.__name__}' finished. ---")

        # Return the result of the original function call.
        return result

    # The decorator returns the wrapper function.
    return wrapper


---

## 3. Applying Decorators

### 3.1. Method 1: The Manual Way
- We are manually "decorating" our function by passing it to the decorator.

In [3]:
def say_hello(name):
    print(f"Hello, {name}!")

# The `decorated_say_hello` variable now holds the `wrapper` function.
decorated_say_hello = logger_decorator(say_hello)
decorated_say_hello("Alice")

--- Calling function: 'say_hello' ---
Hello, Alice!
--- Function 'say_hello' finished. ---


### 3.2. Method 2: The Pythonic Way with `@` Syntactic Sugar
- The `@` symbol is just a cleaner, more readable way to do the exact same thing as above.

In [4]:
# `@logger_decorator` is equivalent to `say_goodbye = logger_decorator(say_goodbye)`
@logger_decorator
def say_goodbye(name):
    print(f"Goodbye, {name}!")

say_goodbye("Bob")

--- Calling function: 'say_goodbye' ---
Goodbye, Bob!
--- Function 'say_goodbye' finished. ---



---

## 4. Preserving Metadata with `functools.wraps`
- Problem: Decorators replace the original function with a wrapper. This can hide the original function's name, docstring, and other metadata.
- Solution: Use `@functools.wraps` inside your decorator.

In [5]:
import time
import functools  # Needed for functools.wraps

def timer_decorator(original_function):
    """A decorator to measure the execution time of a function."""

    @functools.wraps(original_function)  # This is the key!
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = original_function(*args, **kwargs)
        end_time = time.perf_counter()
        print(
            f"Function '{original_function.__name__}' took {end_time - start_time:.4f} seconds."
        )
        return result

    return wrapper

In [6]:
@timer_decorator
def process_data(size):
    """A sample function that simulates some work."""
    print(f"Processing {size} data points...")
    time.sleep(size / 1000)  # Simulate work by sleeping
    return "Done"

In [7]:
# Check the function's metadata
print(
    f"Function name: {process_data.__name__}"
)  # Without @wraps, this would be 'wrapper'
print(f"Docstring: {process_data.__doc__}")  # Without @wraps, this would be None

# Call the decorated function
process_data(1200)

Function name: process_data
Docstring: A sample function that simulates some work.
Processing 1200 data points...
Function 'process_data' took 1.2047 seconds.


'Done'


---

## 5. Decorators with Arguments
- To create a decorator that accepts its own arguments, you need an extra layer of nesting.
- It's a function that takes arguments and *returns a decorator*.

In [8]:
def repeat(num_times):
    """A decorator factory. Returns a decorator that repeats a function call."""

    def decorator_repeat(original_function):
        @functools.wraps(original_function)
        def wrapper(*args, **kwargs):
            total_result = None
            for _ in range(num_times):
                total_result = original_function(*args, **kwargs)
            return total_result

        return wrapper

    return decorator_repeat

In [9]:
# The `@` syntax now calls `repeat(3)`, which returns the actual decorator.
@repeat(num_times=3)
def greet(name):
    """Greets a person."""
    print(f"Hello again, {name}!")

greet("Charlie")

Hello again, Charlie!
Hello again, Charlie!
Hello again, Charlie!



---

**Next:** [Type Hints](./24_type_hints.ipynb)